# GRIDS permission plugin: live demo on Colab
Starts the chat demo from `plugin/`: an assistant (an Ollama model on the GPU) works on AgentDojo's **simulated** mailbox,
calendar and drive, and every action it proposes passes the **Wall** and then **laya-cybersec** (`TextCortex/laya-cybersec`,
322M parameters) in the Gatekeeper slot, which answers ALLOW, ASK or BLOCK (the team's rotated-option method). With
**permission memory** on (the repository's scoped grants), "Approve and remember" stores an exact permission. The page
shows each decision, the memory and the compute each part used.

**Before you start:** Runtime → Change runtime type → a GPU (T4 is enough). Run the cells in order. The last setup cell prints
a public `gradio.live` link that anyone can open while this session runs; nothing real is read or sent.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/AtharvaThorat/permission-that-learn.git"
BRANCH = "11_Nov_Plugin_Cybersec_Memory"   # main, once merged
MODEL = "qwen2.5:7b"                      # Ollama model that drives the assistant; must support tool calling
LAYA_DEVICE = "cpu"                       # laya-cybersec on the CPU, as the project intends; "cuda" is faster

WORK, ENV, CACHE = "/content/permission-that-learn", "/content/plugin-env", "/content/hf-cache"
PY = f"{ENV}/bin/python"

## 2. Helpers
Run once; no changes needed.

In [ ]:
import os, subprocess, time, urllib.request


def sh(cmd, check=True, cwd=None):
    # Run a shell command, stream its output, stop on failure.
    process = subprocess.Popen(cmd, shell=True, cwd=cwd or (WORK if os.path.isdir(WORK) else "/content"),
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="")
    process.wait()
    if check and process.returncode:
        raise RuntimeError(f"Command failed ({process.returncode}): {cmd}")

## 3. Code and Python environment
A separate environment with the repository's Laya pins plus Gradio and MCP (about 3 minutes).

In [ ]:
if not os.path.isdir(WORK):
    sh(f"git clone --branch {BRANCH} {REPO_URL} {WORK}", cwd="/content")
sh(f"git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}")
print("Commit:", subprocess.check_output(["git", "-C", WORK, "rev-parse", "HEAD"], text=True).strip())
sh("pip install -q uv")
if not os.path.exists(PY):
    sh(f"uv venv {ENV} --python 3.12 --seed -q")
sh(f"uv pip install --python {PY} -q -r environments/laya.txt -r plugin/requirements.txt")

## 4. Start Ollama and download the assistant model

In [ ]:
if subprocess.run("which ollama", shell=True, capture_output=True).returncode:
    sh("apt-get update -qq && apt-get install -y -qq zstd pciutils lshw")
    sh("curl -fsSL https://ollama.com/install.sh | sh")
ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "a"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2)
        break
    except OSError:
        time.sleep(1)
sh(f"ollama pull {MODEL} 2>&1 | tail -n 2")

## 5. Launch the demo
Downloads laya-cybersec once (about 680 MB, pinned revision), loads it, and starts the page. Open the printed link.
Remembered permissions are kept in `/content/permissions.sqlite` for this Colab session.
Each scenario in the page explains what it shows; **Start / reset Workspace** applies a new scenario or protection level.

In [ ]:
log = open("/content/demo.log", "w")
demo = subprocess.Popen([PY, "-u", "-m", "grids_plugin.app", "--model", MODEL, "--laya-cache", CACHE,
                         "--laya-device", LAYA_DEVICE, "--memory-store", "/content/permissions.sqlite", "--share"], cwd=f"{WORK}/plugin", stdout=log, stderr=subprocess.STDOUT,
                        env={**os.environ, "GRADIO_ANALYTICS_ENABLED": "False", "PYTHONUNBUFFERED": "1"})
url = None
for _ in range(300):
    text = open("/content/demo.log").read()
    url = next((word for word in text.split() if "gradio.live" in word), None)
    if url or demo.poll() is not None:
        break
    time.sleep(2)
print(f"Open the demo: {url}" if url else "No link yet; the end of the log:\n" + text[-3000:])

## 6. Test on the team dataset (optional)
Runs laya-cybersec on all 150 rows of Siddharth's gated dataset `siddharth-magesh/grids-laya-permission-v1` and prints
the report. Needs only cells 1–3. Nothing is executed.

**Once per Google account:** accept the dataset's terms on Hugging Face, then open the key icon (Secrets) in Colab's
left sidebar, add a secret named `HF_TOKEN` with your Read token, and allow this notebook access when asked.

In [ ]:
from google.colab import userdata
token = userdata.get("HF_TOKEN")
output = f"/content/eval/run-{time.strftime('%Y%m%d-%H%M%S')}"
evaluation = subprocess.run([PY, "-u", "-m", "grids_plugin.evaluate", "--laya-cache", CACHE, "--data", "/content/data",
                             "--output", output, "--laya-device", LAYA_DEVICE], cwd=f"{WORK}/plugin",
                            env={**os.environ, "HF_TOKEN": token, "PYTHONUNBUFFERED": "1"})
if evaluation.returncode == 0:
    print(open(f"{output}/report.md").read())
    print(f"Files in {output}: report.md, summary.json, predictions.jsonl")

### 6b. Replay the dataset in its own environments (optional)
Rebuilds each row's simulated Workspace, Slack or Banking (with its injected text), sends the row's action through the
whole plugin (Wall, laya-cybersec, memory, executor) and checks whether an attack worked. Needs cell 6's token; about
5 minutes.

In [ ]:
from google.colab import userdata
output = f"/content/replay/run-{time.strftime('%Y%m%d-%H%M%S')}"
replay = subprocess.run([PY, "-u", "-m", "grids_plugin.replay", "--laya-cache", CACHE, "--data", "/content/data",
                         "--output", output, "--laya-device", LAYA_DEVICE], cwd=f"{WORK}/plugin",
                        env={**os.environ, "HF_TOKEN": userdata.get("HF_TOKEN"), "PYTHONUNBUFFERED": "1",
                             "PYTHONUTF8": "1"})
if replay.returncode == 0:
    print(open(f"{output}/report.md").read())

## 7. Stop

In [ ]:
for process in ("demo", "ollama"):
    if process in globals():
        globals()[process].terminate()
print("Stopped.")

## 8. If something failed
The end of the demo and Ollama logs.

In [ ]:
print(open("/content/demo.log").read()[-4000:])
print(open("/content/ollama.log").read()[-2000:])